In [1]:
import pandas as pd
import numpy as np

csv_path = "spottolerance_offline_shuntserve_unittest8b_20260320_0918.csv"
df = pd.read_csv(csv_path)
print(f"Total requests: {len(df)}")
print(f"Successful: {df['Success'].sum()}")
print(f"SendCount distribution:\n{df['SendCount'].value_counts().sort_index()}")
print(f"\nColumns: {list(df.columns)}")
df.head(10)

Total requests: 1050
Successful: 1050
SendCount distribution:
SendCount
1    632
2    360
3     57
4      1
Name: count, dtype: int64

Columns: ['RequestID', 'ArrivalTime', 'CompletionTime', 'InputTokens', 'OutputTokens', 'Latency', 'TTFT', 'TPOT', 'Success', 'QueueingDelay', 'SendCount']


,RequestID,ArrivalTime,CompletionTime,InputTokens,OutputTokens,Latency,TTFT,TPOT,Success,QueueingDelay,SendCount
0,786,1.773999e+09,1.773999e+09,1208,413,142.858779,7.509990,0.328516,True,260.556860,2
1,191,1.773998e+09,1.773999e+09,1113,531,135.566628,1.536090,0.252888,True,0.002783,1
2,276,1.773998e+09,1.773999e+09,1095,379,162.175442,38.249505,0.327846,True,24.091524,2
3,956,1.773999e+09,1.773999e+09,1133,429,139.397900,4.041360,0.316254,True,308.076174,2
4,361,1.773998e+09,1.773999e+09,1132,414,132.999055,4.328058,0.311552,True,57.549416,2
5,787,1.773999e+09,1.773999e+09,379,62,30.858633,12.422827,0.302226,True,260.247710,1
6,192,1.773998e+09,1.773998e+09,1313,170,38.849671,1.494201,0.221038,True,0.000800,1
7,277,1.773998e+09,1.773999e+09,1079,419,172.807207,36.411550,0.326305,True,23.753706,2
8,957,1.773999e+09,1.773999e+09,1199,396,131.674504,6.485892,0.316933,True,308.908046,2
9,362,1.773998e+09,1.773999e+09,1088,386,129.955805,12.486583,0.305115,True,57.614718,2


## 검증 1: QueueingDelay >= 0 (모든 request)

In [2]:
negative_qd = df[df['QueueingDelay'] < 0]
print(f"QueueingDelay < 0: {len(negative_qd)} / {len(df)}")
if len(negative_qd) > 0:
    print("FAIL — negative queueing delays found:")
    print(negative_qd[['RequestID', 'QueueingDelay', 'Latency', 'SendCount']].head(10))
else:
    print("PASS")

QueueingDelay < 0: 0 / 1050
PASS


## 검증 2: CompletionTime - ArrivalTime == Latency + QueueingDelay (정합성)

In [3]:
df['TotalTime'] = df['CompletionTime'] - df['ArrivalTime']
df['ReconstructedTotal'] = df['Latency'] + df['QueueingDelay']
df['Diff'] = abs(df['TotalTime'] - df['ReconstructedTotal'])

print(f"Max diff: {df['Diff'].max():.10f}")
print(f"Mean diff: {df['Diff'].mean():.10f}")
if df['Diff'].max() < 0.001:
    print("PASS — Latency + QueueingDelay == CompletionTime - ArrivalTime")
else:
    print("FAIL")
    print(df[df['Diff'] > 0.001][['RequestID', 'TotalTime', 'ReconstructedTotal', 'Diff']].head(10))

Max diff: 0.0000012264
Mean diff: 0.0000003920
PASS — Latency + QueueingDelay == CompletionTime - ArrivalTime


## 검증 3: Migration request의 gap (Latency - TTFT - TPOT*(OutputTokens-1)) >= 0

In [4]:
# TPOT = (Latency - TTFT) / (OutputTokens - 1), so TTFT + TPOT*(OutputTokens-1) = Latency for non-migration
# For migration requests, token_time < Latency (gap = halt detection time)
# We reconstruct token_time from TTFT and TPOT

migrated = df[df['SendCount'] > 1].copy()
non_migrated = df[df['SendCount'] == 1].copy()

print(f"Migrated requests: {len(migrated)}")
print(f"Non-migrated requests: {len(non_migrated)}")

# For non-migrated: TTFT + TPOT * (OutputTokens-1) should == Latency
non_migrated['TokenTime'] = non_migrated['TTFT'] + non_migrated['TPOT'] * (non_migrated['OutputTokens'] - 1)
non_migrated['GapToLatency'] = non_migrated['Latency'] - non_migrated['TokenTime']
print(f"\n--- Non-migrated ---")
print(f"Gap (Latency - TokenTime) stats:")
print(non_migrated['GapToLatency'].describe())

# For migrated: gap should be positive (unaccounted halt detection time)
migrated['TokenTime'] = migrated['TTFT'] + migrated['TPOT'] * (migrated['OutputTokens'] - 1)
migrated['GapToLatency'] = migrated['Latency'] - migrated['TokenTime']
print(f"\n--- Migrated ---")
print(f"Gap (Latency - TokenTime) stats:")
print(migrated['GapToLatency'].describe())

negative_gap = migrated[migrated['GapToLatency'] < -0.01]
if len(negative_gap) == 0:
    print("\nPASS — all migrated requests have gap >= 0")
else:
    print(f"\nFAIL — {len(negative_gap)} migrated requests have negative gap:")
    print(negative_gap[['RequestID', 'Latency', 'TokenTime', 'GapToLatency', 'SendCount']].head(10))

Migrated requests: 418
Non-migrated requests: 632

--- Non-migrated ---
Gap (Latency - TokenTime) stats:
count    632.000000
mean      -0.000002
std        0.000065
min       -0.000237
25%       -0.000032
50%        0.000000
75%        0.000029
max        0.000232
Name: GapToLatency, dtype: float64

--- Migrated ---
Gap (Latency - TokenTime) stats:
count    4.180000e+02
mean     6.052632e-07
std      1.099778e-04
min     -2.860000e-04
25%     -7.875000e-05
50%     -1.000000e-06
75%      7.700000e-05
max      2.510000e-04
Name: GapToLatency, dtype: float64

PASS — all migrated requests have gap >= 0


## 검증 4: 기본 통계 요약

In [5]:
print("=== Overall ===")
print(f"Latency    — mean: {df['Latency'].mean():.2f}s, median: {df['Latency'].median():.2f}s, p99: {df['Latency'].quantile(0.99):.2f}s")
print(f"TTFT       — mean: {df['TTFT'].mean():.2f}s, median: {df['TTFT'].median():.2f}s, p99: {df['TTFT'].quantile(0.99):.2f}s")
print(f"TPOT       — mean: {df['TPOT'].mean():.4f}s, median: {df['TPOT'].median():.4f}s")
print(f"QueueDelay — mean: {df['QueueingDelay'].mean():.2f}s, median: {df['QueueingDelay'].median():.2f}s")

print(f"\n=== Non-migrated (SendCount=1) ===")
nm = df[df['SendCount'] == 1]
print(f"Count: {len(nm)}")
print(f"Latency    — mean: {nm['Latency'].mean():.2f}s, median: {nm['Latency'].median():.2f}s")
print(f"QueueDelay — mean: {nm['QueueingDelay'].mean():.2f}s, median: {nm['QueueingDelay'].median():.2f}s")

print(f"\n=== Migrated (SendCount>1) ===")
m = df[df['SendCount'] > 1]
print(f"Count: {len(m)}")
print(f"Latency    — mean: {m['Latency'].mean():.2f}s, median: {m['Latency'].median():.2f}s")
print(f"QueueDelay — mean: {m['QueueingDelay'].mean():.2f}s, median: {m['QueueingDelay'].median():.2f}s")

=== Overall ===
Latency    — mean: 85.82s, median: 81.88s, p99: 201.76s
TTFT       — mean: 8.85s, median: 7.05s, p99: 38.50s
TPOT       — mean: 0.3159s, median: 0.2993s
QueueDelay — mean: 148.64s, median: 145.03s

=== Non-migrated (SendCount=1) ===
Count: 632
Latency    — mean: 57.21s, median: 45.87s
QueueDelay — mean: 135.05s, median: 118.06s

=== Migrated (SendCount>1) ===
Count: 418
Latency    — mean: 129.09s, median: 137.48s
QueueDelay — mean: 169.18s, median: 174.54s


## 검증 5: JSON 결과와 CSV 비교

In [6]:
import json

with open("../offline_shuntserve.json") as f:
    js = json.load(f)

success = df[df['Success'] == True]

def compare(name, json_val, csv_val, unit=""):
    diff = abs(json_val - csv_val)
    pct = (diff / json_val * 100) if json_val != 0 else 0
    status = "PASS" if pct < 1.0 else ("WARN" if pct < 5.0 else "FAIL")
    print(f"  {status} {name:30s} JSON={json_val:12.4f}  CSV={csv_val:12.4f}  diff={pct:.2f}%")

print(f"JSON completed: {js['completed']}, CSV successful: {len(success)}")
print()

# Basic counts
compare("completed", js['completed'], len(success))
compare("total_input", js['total_input'], success['InputTokens'].sum())
compare("total_output", js['total_output'], success['OutputTokens'].sum())

print()

# Latency metrics (JSON is in ms, CSV is in seconds)
compare("mean_ttft_ms", js['mean_ttft_ms'], success['TTFT'].mean() * 1000)
compare("median_ttft_ms", js['median_ttft_ms'], success['TTFT'].median() * 1000)
compare("mean_tpot_ms", js['mean_tpot_ms'], success['TPOT'].mean() * 1000)
compare("median_tpot_ms", js['median_tpot_ms'], success['TPOT'].median() * 1000)
compare("mean_e2el_ms", js['mean_e2el_ms'], success['Latency'].mean() * 1000)
compare("median_e2el_ms", js['median_e2el_ms'], success['Latency'].median() * 1000)

print()

# Throughput
csv_duration = success['CompletionTime'].max() - success['ArrivalTime'].min()
compare("benchmark_duration", js['benchmark_duration'], csv_duration)
compare("request_throughput", js['request_throughput'], len(success) / csv_duration)
compare("output_throughput", js['output_throughput'], success['OutputTokens'].sum() / csv_duration)

JSON completed: 1050, CSV successful: 1050

  PASS completed                      JSON=   1050.0000  CSV=   1050.0000  diff=0.00%
  PASS total_input                    JSON= 757821.0000  CSV= 757821.0000  diff=0.00%
  PASS total_output                   JSON= 265252.0000  CSV= 265252.0000  diff=0.00%

  PASS mean_ttft_ms                   JSON=   8854.2516  CSV=   8854.2516  diff=0.00%
  PASS median_ttft_ms                 JSON=   7053.2384  CSV=   7053.2385  diff=0.00%
  PASS mean_tpot_ms                   JSON=    315.9244  CSV=    315.9244  diff=0.00%
  PASS median_tpot_ms                 JSON=    299.2953  CSV=    299.2955  diff=0.00%
  PASS mean_e2el_ms                   JSON=  85823.4863  CSV=  85823.4863  diff=0.00%
  PASS median_e2el_ms                 JSON=  81877.4723  CSV=  81877.4725  diff=0.00%

  PASS benchmark_duration             JSON=    720.6247  CSV=    719.8032  diff=0.11%
  PASS request_throughput             JSON=      1.4571  CSV=      1.4587  diff=0.11%
  PASS o